# RR BENCHMARK PHASE B1: build and FREEZE the query signatures (NO SCORING)
Pre-registration SHA-256 7eee1779...c9a2. This notebook computes **no connectivity score**.
It (1) checks every drug ID by molecular formula, (2) builds the disease-vs-control signature for rows 8, 10, 12, 14/15, 17, 18 with the control-2 statistic (k=150 up/150 down, landmark genes only), (3) runs diagnostics on rows 6 and 13, whose data are not in the series matrix, and (4) freezes what it built with a SHA-256.
Run: hamburger > Runtime > Run all. Paste back everything between `=== PHASE B1 REPORT BEGIN ===` and `=== PHASE B1 REPORT END ===`.

In [ ]:
# CELL 1: setup
from google.colab import drive
drive.mount("/content/drive")
import os,subprocess,sys
W="/content/drive/MyDrive/RitschelResearch/benchmark"; os.chdir(W)
subprocess.run([sys.executable,"-m","pip","install","-q","rdkit"],check=False)
import numpy as np,pandas as pd,gzip,re,urllib.request,io,base64,hashlib,json,collections,tarfile
g=pd.read_csv("geneinfo_beta.txt",sep="\t")
LMG=g.loc[g.feature_space=="landmark",["gene_id","gene_symbol"]].copy(); LMG["gene_id"]=LMG.gene_id.astype(int)
genes=np.load("libA_genes.npy"); GI={int(x):i for i,x in enumerate(genes)}
SYM2E={s.upper():int(e) for e,s in zip(LMG.gene_id,LMG.gene_symbol)}
assert len(genes)==978 and set(int(x) for x in genes)==set(LMG.gene_id), "LIBRARY GENES != LANDMARKS"
REPORT=[]; R=lambda *a: (print(*a), REPORT.append(" ".join(str(x) for x in a)))
R("library genes",len(genes))

In [ ]:
ROST_B64="H4sICAnCvmoCA3Jvc3QuY3N2AIWd3c4jRxVF73mWQfL5q3POA6DcgITEE6AwQkhAUIZIPD52rrodrz3KTVT76/a3V1WX1/S4Pd++/vyPr9++/P3bv778/eeffvnP7374yx/sEZFffvjLn/zYY86XH//67etvxxvG58uPP/37vz//9M/fRvv5kH3AuOGp1uGQgPGE8YLxwy8NxXdgfOlU9vhc3B4G486nCjgkYbxg/MB4w/jAOLc2aG3Q2hzGg18CWhu0tsOnao6gu31e5+ZQ3KG483Q7TLdDcYfiDtPt3NqhtUPrgNYBrQOmO6Dy8//oVw1oHdA6uHUMR1A8oXhC8YTiyes8YboTiiev84QLPGG6E1oXtC7czK2geMGMF894QfGCGS9oXdC6eFs7UPzAdB++ug8UPzDdp/hUUPxA8cPr/MCMNxRvnvGGGW8o3jzjDTPeULz5Gm+Y9IbiA8UHZnyg9UDr4dYDrcHWDGzN2NYMbM3A1myhNaiaLW9rYGsGtmZsawa2ZmBrxrbmYGsOtuZsaw625mBr/ig+1eGo4WwD459n3EHYHITNQdjcoDXYmoOtuR0Yx0vbQdUcVM1fqganAltzh9YefCooDrbmznPtMNdga+68zkHYHITNQdgchM1Z2ByEzUHYnIXNA4qDrXnyjIOwOQibs7A5CJuDsDkLm4OwOQibJ884OJsXFAdhcxA2Z2FzEDYHYfPiGQdn84IZB2FzEDY/0BpszcHW/EDlw3MNtuYHKoOqOaiaN1RufgsDVfOG1uBpDp7m7GkOnubgaT58aYOqOaiaD1/aA8VB1Xx4usHWfKA4qJqDqjmomoOqOauag6o5qJqzqjmomoOq+X5uHeBpAZ4WD4dxbB3gaQF31YIlLUDSAiQt+MZagKcFeFqApwV4WljyS0NxULVgVQtQtQBVC1a1AFULULVgVQtQtQBVC1a1AFULULVgVQtQtQBVi7jv55Hn+ab2zGLP80peEdZDhaZC9ZoVKkwVlgqPCluFo8LLErwnV5d4SwwTxyQwSUwKk4NJYzKYIINGBo0MGhk0Mmhk0MigkUEjg0YGjQwGGQwyGGQwyGCQwSCDQQaDDAYZDDJYZLDIYJHBIoNFBosMFhksMlhksMDgqSAPTAyTC4Pz/O/3P/z5j/W8qJ55PqquovApHh2vjK/q8Ck2Hevf/OoTn+LUcck47u89n37Cv/sT8d2fyO/+RH33J853f6K/+xPz3Z+4v4t3nuxX9sRgUSo8DxWaCl2FocJUYanwqLBVOCpUhFoRakWoFaFWhFoRakXo+p73ljQmg8lScn3Pe0sME8ckMElMChNkMMhgkMEgg0UGiwwWGSwyWGSwyGCRwSKDRQZLDPLxwMQwcUwCk8SkMDmYNCaDCTIwZGDIwJCBIQNDBoYMDBkYMjBkYMjAkYEjA0cGjgwcGTgycGTgyMCRgSODQAaBDAIZBDIIZBDIIJBBIINABoEMEhkkMkhkkMggkUEig0QGiQwSGSQyKGRQyKCQQSGDQgaFDAoZFDIoZFDI4CCDgwwOMjjI4CCDgwwOMjjI4CCDgwwaGTQyaGTQyKCRQSMD9MRET0z0xERPTPTERE9M9MRET0z0xERPTPTERE9M9MRET0z0xERPTPTERE9M9MRET0z0xERPTPTERE8s9MRCTyz0xEJPLPTEQk8s9MRCTyz0xEJPLPTEQk8s9MRCTywTf3r89c4Kh0eFrcJRofjzdaE5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjqbuQpe5ClroLWeouZN3vQvrzZZ7XwzMce9TcP4V6D/fxUKGp0FUYKkwVlgqPCluFo0JFyBShyxvse+KYBCaJSWFyMGlMBpOl5PJ2+p4gA0cGjgwcGTgycGTgyMCRgSODQAaBDAIZBDIIZBDIIJBBIINABoEMEhkkMkhkkMggkUEig0QGiQwSGaTaFUrtCqX2zVL7Zql9s9S+WWrfLLVvlto3S+2bpQgdRegoQkcROorQUYSOInQUoaMIqffePYpQK0KtCLUi1IpQK0KtCLUi1IpQK0KtCI0iNIrQKEKjCI0iNIrQKEKjCI0iNIrQKkKrCK0itLgvL+7Li/vy4r68uC8v7ssL703n+kGZ98QwcUwCk8SkMDmYNCaDCTIwZGDIwJCBIQNDBoYMDBkYMjBkYMjAkYEjA0cGjgwcGTgycGTgyMCRgSODQAaBDAIZBDIIZBDIIJBBIINABoEMEhkkMkhkkMggkUEig0QGiQwSGSQyKGRQyKCQQSGDQgaFDAoZFDIoZFDI4CCDgwwOMjjI4CCDgwwOMjjI4CCDgwwaGTQyaGTQyKCRQSODRgaNDBoZNDIYZDDIYJDBIINBBoMMBhkMMhhkMMhgkcEig0UGiwwWGSw79Hn7rqT3sFU4KmSHPibugR4T90CPiXugx8Q90GPiHugxcQ/0mLgHegxd0tAlDV3S0CUNXdLQJQ1d0tAlDV3S0CUNXfL6HUtRL+9+BS9dbBifL1//9+M/f/nb17/9NtvPx1yY3ccNxh3GA8aTf6cLrfv4EcdAdxPdDbo7dHfo7tDdofvFt+/jxb+ri+4O3R3WiUPvgN5P08bXDuj+NG0+Rsx9CAYX376PtzgGGAQwSGCQMPcJ/VP0T5j/hHWf0Dth3hM6J3Qu6FzQuaBzwXov6FvQt8RaLzHXBb0Lej99Gs91oPvTp/kY6H+g/4H+B+b7wHwfsccd6N4w5w29G+a8oXOLa7yhd4t5b+jeMOcNvQd6D/Qe6D3Qe2CuBzoPzPWIdT5ivgd6r1jrC90Xui90X+i+0H3FfC/M98J8P82YznX98oP7uMG4w3jAeMJ4wfiB8YbxEd0+z3eAvwX4W4C/BfhbGHQGd7t+6cF9HDqD4wY4W4CzBTjb6+sOkKmHyFJk0F24W4C7Bbjb64sP8FzgbyH8LcDfImDeA+Y9oDs4WwT0Bl8L8LVI3tcCnC3A2UI4W4CzBThbpJjzbJGJaz3F3IPDBThcgMMFOFyAw0WV+J1g7oXDBTjc6yldPOZA9yPWvfC4AI97PZjLxwgOR6wFcLo4wAF87vUYLr4GOF2A072eveVzwToAp7s+cHsfh94t1j84XYDTBTjd60FbfA3wugCvC/C6AK+7PmJ7H4f5Bp+7Plx7H4fOKzqDzwX4XIDPXR+qvY9DZ3C5EC73eqCWs8/dE3wuwecSfC7B5xJ87vUYLf+uA8d8nu/XQ7R4LnC6BKdLcLoU9+QSvC7B6xK8LsHrErwuwesSvC7hXlzCvbiEe3EJLnd9WPY+Dn3B4xLuwSU43PUR2fs49AV/S/C3BH9L8LcEf3s9FItrKMR1LTwuweNeD8byMdAfPC7B4xLuvSXce0vhbwn33xLcLcHdEtwtwd0S3C3h/luCt6Xwtrx/wPIeiWkXt+BS6FveP1t5j0IclnxYicOOyJpPyUyOYHL/POU9Ekiakdw/SnmPxJ7fAsn9Y5T3iIkIu3s9JIvZiGVy//zkPRIbxDCT4VVy/+DkPWIi4mZeDi+S+ycmb5G4nfd6apYzXiQrrpsVq2QZibjBl8urZMUqWWRSQg1fT9Fy5iILkSX/KiUOO3xYi8OGD2Mm9uBIIDHnw4KjFGcsPoyB3J+prX7dHnpGr29emovdvgVBQVJQFBwKmoKhYCG4GO5bYBRQc6fmTs2dmjs1d2ru1NypeTx4du9fYfeWuchCZCmyEtkRmVih9y+me8uWsxRcUnBJwSUFlxRcUnBJwSUFlxRcUnApwaUElxJcSnApwaUElxJcSnApwaUElyO4HMHlCC5HcDmCyxFcjuByBJcjuBzBpQWXFlxacGnBpQWXFlxacGnBpQWXpk13BJQRUEZAGQFlBJQRUEZAGQFlBJQRi2UFlxVcVnBZwWUFlxVcVnBZwWUFl2Uub0/Wv2UmMhdZiCxFViI7ImuRjcgEFxNcTHAxwcUEFxNcTHAhhV1S2DVBxAQRF0RcEHFBxAURF0RcEHGxUlysFBLcJcG9Pmj/FhgFTkFQkBQUBbQiglZEUPOg5knNk5onNU9qntQ8qXlS86TmSc2Tmhc1L2pe1LyoeVHzouZFzYuaFzUvan6ouXDQFQ56/Scf3gLqL+xzhX2usM8V9rnCPlfY5wr7XGGfK+xzhX2usM8V9rnCPlfY59vT8PdMCOgKAV0hoCsEdIWArhDQFQK6QkBXCOgKAV0hoCsE9PrPRLwFdPksXT5L28fS9rG0fSxtH/tx+5jH9an3t8AocAqCgqSgKDgUNAU0/6+MyqNNvjITmYssRJYiK5ERDiMcn2+IvgJi4bQQXIBwWgtOa8FpLTitBafyaI2vjPo79Q/qH3QhBJUPKh9UPqh8UPmgmQ9qHtQ8qXlS86TmSc2Tmic1T2qe1DypeVLzouZFzYuaFzUval7UvKh5UfOi5kXNDzU/1PxQ80PNDzU/1PxQ80PNDzU/1LypeVPzpuZNzZuaNzVvat7UvKl5U/Oh5kPNh5oPNR9qPtR8qPlQ86HmQ82Xmi81X2q+1Hyp+VLzpeZLzZeak94Z6Z2R3hnpnZHeGemdkd4Z6Z2R3l0fQH8LqLlRc6PmRs2Nmhs1N2pOJmdkckYmZ2RyRiZnTs2dfdY8RHbz2dP267/99fr+4cu+fBteOuCyLd+GDQ/wzwcEHpCfDyg84Hw+oPGAz6UbS8/n0oOl53PpwdLzufRg6flcerD0fC49WHo/l14svZ9LX/6a5/+8jQXXO44AAA=="
# CELL 2: rosters embedded from ledger/rosters_2026-10-01 (built and verified 2026-10-01)
raw=gzip.decompress(base64.b64decode(ROST_B64))
assert hashlib.md5(raw).hexdigest()=="ce6b1b2f36a4fff4c7034c4d99d4d280","ROSTER TRANSFER CORRUPTED"
ROS=pd.read_csv(io.BytesIO(raw))
EXP={"GSE174647":(182,28),"GSE256539":(152,68),"GSE10334":(183,64),"GSE35007":(190,31),"GSE134662":(44,10),
     "GSE57338":(177,136),"GSE6710":(13,13),"GSE16161-GPL570":(9,9)}
for s,(c,k) in EXP.items():
    d=ROS[ROS.series==s].group.value_counts()
    assert d.get("case",0)==c and d.get("control",0)==k, ("ROSTER COUNT MISMATCH",s,dict(d))
R("rosters OK",len(ROS),"rows, all 8 series match EXPECT")

In [ ]:
# CELL 3: drug IDs, formula check (largest fragment, so salts and water drop out).
# RULE fixed before scoring: within each drug, keep IDs whose formula equals the drug's reference formula,
# reference = formula of the ID whose cmap_name equals the drug name and that has the most library signatures.
from rdkit import Chem
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
ci=pd.read_csv("compoundinfo_beta.txt",sep="\t",dtype=str).fillna("")
D=pd.read_csv("drugs_resolved_phaseA.csv",dtype={"pert_id":str})
D["drug"]=D.query_name.str.replace("(skeleton)","",regex=False).replace({"cyclosporin a":"cyclosporin-a","cyclosporine":"cyclosporin-a","ciclosporin":"cyclosporin-a"})
def formula(pid):
    sm=ci.loc[ci.pert_id==pid,"canonical_smiles"]
    sm=[x for x in sm if x]
    if not sm: return "NO_SMILES"
    m=Chem.MolFromSmiles(sm[0])
    if m is None: return "BAD_SMILES"
    frags=Chem.GetMolFrags(m,asMols=True); big=max(frags,key=lambda x:x.GetNumHeavyAtoms())
    return CalcMolFormula(big)
D["formula"]=D.pert_id.map(formula)
keep=[]
for (row,drug),grp in D.groupby(["row","drug"]):
    ref=grp.sort_values("n_lib_sigs",ascending=False).iloc[0].formula
    for _,x in grp.iterrows():
        ok=(x.formula==ref) and not x.excluded and x.n_lib_sigs>0
        keep.append(ok)
        R(f"row {row} {drug:15s} {x.pert_id} {x.formula:14s} ref {ref:14s} lib_sigs {x.n_lib_sigs:3d} -> {'KEEP' if ok else 'DROP'}")
D["keep"]=keep
F=D[D.keep][["row","drug","pert_id","formula","n_lib_sigs","n_cells"]]
F.to_csv("drugs_frozen.csv",index=False)
for (row,drug),grp in F.groupby(["row","drug"]):
    assert grp.n_cells.max()>=2, ("G2b FAIL",row,drug)
R("drugs frozen:",len(F),"IDs over",F.groupby(['row','drug']).ngroups,"drugs")

In [ ]:
# CELL 4: loaders and the control-2 statistic (copied from control 2, unchanged)
def mpath(s): return s+"_series_matrix.txt"
def get_matrix(s):
    f=mpath(s)
    if not os.path.exists(f):
        g_,p_=(s.split("-")+[None])[:2]
        name=f+".gz"; u="https://ftp.ncbi.nlm.nih.gov/geo/series/%snnn/%s/matrix/%s"%(g_[:-3],g_,name)
        open(f,"wb").write(gzip.decompress(urllib.request.urlopen(u,timeout=300).read()))
    return f
def read_matrix(s):
    hdr=None; rows=[]; inb=False; meta=[]
    for line in open(get_matrix(s),encoding="utf-8",errors="replace"):
        if line.startswith("!series_matrix_table_end"): break
        if inb:
            p=line.rstrip("\n").split("\t")
            if hdr is None: hdr=[x.strip().strip('"') for x in p]
            else: rows.append(p)
        else: meta.append(line)
        if line.startswith("!series_matrix_table_begin"): inb=True
    return hdr[1:],rows,meta
def annot_map(plat):
    f=plat+".annot.gz"
    if not os.path.exists(f):
        u="https://ftp.ncbi.nlm.nih.gov/geo/platforms/%snnn/%s/annot/%s.annot.gz"%(plat[:-3],plat,plat)
        try: urllib.request.urlretrieve(u,f)
        except Exception as e: return None
    p2e={}; inb=False; ige=None
    for raw in gzip.open(f,"rt",encoding="utf-8",errors="replace"):
        if raw.startswith("!platform_table_end"): break
        if inb:
            q=raw.rstrip("\n").split("\t")
            if ige is None: ige=[i for i,c in enumerate(q) if "Gene ID" in c]; continue
            if ige and len(q)>ige[0]:
                x=q[ige[0]].split("///")[0].strip()
                if x.isdigit(): p2e[q[0]]=int(x)
        if raw.startswith("!platform_table_begin"): inb=True
    return p2e
def gene_level(samples,rows,keyfun):
    # PRE-REGISTERED (control 2): median across probes mapping to the same landmark gene
    acc={}
    for p in rows:
        e=keyfun(p[0].strip().strip('"'))
        if e is None or e not in GI: continue
        try: v=np.array([float(y) if y not in ("","null","NA","NaN") else np.nan for y in (x.strip().strip('"') for x in p[1:])],dtype=float)
        except ValueError: continue
        acc.setdefault(GI[e],[]).append(v)
    E=np.full((len(genes),len(samples)),np.nan)
    for gidx,L in acc.items(): E[gidx]=np.nanmedian(np.vstack(L),axis=0)
    return E
def logify(E,tag):
    q=np.nanpercentile(E,99)
    if q>100:
        R(f"  {tag}: 99th pct {q:.0f} > 100 -> linear scale, applying log2(max(x,1)) [RULE fixed before scoring]")
        return np.log2(np.maximum(E,1.0))
    R(f"  {tag}: 99th pct {q:.2f} -> already log scale, unchanged"); return E
def stat(E,a,b,k=150):
    A=E[:,a]; B=E[:,b]
    d=A.mean(1)-B.mean(1)
    pooled=np.sqrt(0.5*(A.var(1,ddof=1)+B.var(1,ddof=1)))
    fl=np.median(pooled[pooled>0]) if np.any(pooled>0) else 1.0
    st=d/np.maximum(pooled,fl*0.1); o=np.argsort(-st,kind="stable")
    return o[:k].copy(),o[-k:].copy()
SIG={}
def build(row,s,E,samples,note=""):
    r=ROS[ROS.series==s].set_index("gsm").group
    keep=~np.isnan(E).any(axis=1)
    cases=[i for i,x in enumerate(samples) if r.get(x)=="case"]; ctrls=[i for i,x in enumerate(samples) if r.get(x)=="control"]
    Ek=E[keep]; idx=np.where(keep)[0]
    up,dn=stat(Ek,cases,ctrls)
    SIG[row]=dict(series=s,n_case=len(cases),n_ctrl=len(ctrls),genes_usable=int(keep.sum()),
                  up=[int(genes[idx[i]]) for i in up],dn=[int(genes[idx[i]]) for i in dn],usable=[int(genes[i]) for i in idx],note=note)
    R(f"row {row} {s}: cases {len(cases)} controls {len(ctrls)} landmark genes usable {int(keep.sum())} of 978 {note}")
    assert len(cases)>0 and len(ctrls)>0, "EMPTY ARM IS FATAL"

In [ ]:
# CELL 5: array rows 10, 12, 14/15, 17, 18
for row,s in [("10","GSE10334"),("12","GSE35007"),("14/15","GSE57338"),("17","GSE6710"),("18","GSE16161-GPL570")]:
    samples,rows,meta=read_matrix(s)
    plat=re.findall(r'!Series_platform_id\t"?([^"\t\n]+)',"".join(meta))
    plat=["GPL570"] if s.endswith("GPL570") else plat
    p2e=annot_map(plat[0]); src=plat[0]
    if p2e is None and plat[0]=="GPL96":
        p2e=annot_map("GPL570"); src="GPL570 annot (HG-U133A probe IDs are a subset of HG-U133 Plus 2) [RULE]"
    assert p2e, ("NO ANNOTATION",s)
    E=gene_level(samples,rows,lambda pid: p2e.get(pid))
    E=logify(E,s)
    build(row,s,E,samples,note=f"annot {src}")

In [ ]:
# CELL 6: row 8 GeoMx (symbols). RULE fixed before scoring: if a donor/patient field exists,
# average to one value per donor within arm (ROIs of one patient are not independent); else per GSM, flagged.
s="GSE256539"; samples,rows,meta=read_matrix(s)
E=gene_level(samples,rows,lambda sym: SYM2E.get(sym.upper()))
E=logify(E,s)
M="".join(meta); ch=[l for l in meta if l.startswith("!Sample_characteristics_ch1")]
fields=collections.OrderedDict()
for l in ch:
    vals=[x.strip().strip('"') for x in l.rstrip("\n").split("\t")[1:]]
    for i,v in enumerate(vals):
        if ":" in v: fields.setdefault(v.split(":",1)[0].strip().lower(),{})[i]=v.split(":",1)[1].strip()
R("row 8 characteristic fields:",list(fields.keys()))
dk=[k for k in fields if re.search(r"patient|donor|subject|individual|case id|sample id",k)]
r8=ROS[ROS.series==s].set_index("gsm").group
if dk:
    key=dk[0]; don=[fields[key].get(i,"NA") for i in range(len(samples))]
    grp=[r8.get(x,"none") for x in samples]
    units=sorted(set((d_,g_) for d_,g_ in zip(don,grp) if g_ in ("case","control")))
    both=set(d_ for d_,g_ in units if g_=="case")&set(d_ for d_,g_ in units if g_=="control")
    R(f"row 8 donor field '{key}': {len(set(don))} donors; donors in BOTH arms: {len(both)}")
    Ea=np.column_stack([np.nanmean(E[:,[i for i,(d_,g_) in enumerate(zip(don,grp)) if d_==u and g_==gg]],axis=1) for u,gg in units])
    ROS=pd.concat([ROS,pd.DataFrame({"series":s+"_donor","gsm":[f"{u}|{gg}" for u,gg in units],"group":[gg for u,gg in units]})])
    build("8","GSE256539_donor",Ea,[f"{u}|{gg}" for u,gg in units],note=f"aggregated to donor via '{key}' from {len(samples)} ROIs")
else:
    build("8",s,E,samples,note="NO donor field found: per-ROI, pseudoreplication FLAGGED")

In [ ]:
# CELL 7: diagnostics for rows 6 and 13 (no signature built here)
def gsm_text(gsm):
    u="https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=%s&targ=self&form=text&view=brief"%gsm
    return urllib.request.urlopen(u,timeout=60).read().decode("utf-8","replace")
for row,s in [("13","GSE134662"),("6","GSE174647")]:
    samples,rows,meta=read_matrix(s)
    R(f"--- row {row} {s}: {len(samples)} samples; first GSM {samples[0]}")
    for gsm in samples[:3]:
        try:
            t=gsm_text(gsm)
            for l in t.splitlines():
                if l.startswith(("!Sample_title","!Sample_supplementary_file","!Sample_relation","!Sample_characteristics_ch1","!Sample_source_name")): R("   ",gsm,l[:160])
        except Exception as e: R("   ",gsm,"FETCH FAILED",e)
s="GSE174647"; f="GSE174647_ST_counts.tar.gz"
if not os.path.exists(f):
    urllib.request.urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE174nnn/GSE174647/suppl/"+f,f)
R("row 6 ST_counts size",os.path.getsize(f))
with tarfile.open(f) as t:
    mem=t.getmembers(); R("row 6 tar members:",len(mem))
    for m in mem[:12]: R("   ",m.name,m.size)
    first=[m for m in mem if m.isfile()][0]; fh=t.extractfile(first); head=fh.read(600)
    try: head=gzip.decompress(head)  # may fail on a partial gzip; fine
    except Exception: pass
    R("row 6 first member head:",repr(head[:400]))
sp="GSE174647_ST_spatial_coord_annotations_spots.csv.gz"
if not os.path.exists(sp): urllib.request.urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE174nnn/GSE174647/suppl/"+sp,sp)
a=pd.read_csv(sp,nrows=5); R("row 6 spot annotation columns:",list(a.columns)); R(a.head(3).to_string())

In [ ]:
# CELL 8: freeze and report
out={"prereg_sha256":"7eee1779246f461b8595eeef8cffd17ee93ead40f6ed36b6af390cd421b7c9a2","k":150,"statistic":"control-2 pooled-sd t with 0.1*median floor",
     "signatures":SIG,"drugs":F.to_dict(orient="records")}
b=json.dumps(out,sort_keys=True).encode(); open("signatures_frozen_B1.json","wb").write(b)
rb=open("signatures_frozen_B1.json","rb").read(); assert rb==b, "FREEZE READ-BACK MISMATCH"
h=hashlib.sha256(rb).hexdigest()
print("=== PHASE B1 REPORT BEGIN ===")
print("\n".join(REPORT))
for row,v in SIG.items(): print(f"SIG row {row}: {v['series']} case {v['n_case']} ctrl {v['n_ctrl']} usable {v['genes_usable']} up[:5] {v['up'][:5]} dn[:5] {v['dn'][:5]}")
print("FROZEN signatures_frozen_B1.json sha256",h)
print("=== PHASE B1 REPORT END ===")